# PubMed Document Retrieval & Cleaning Pipeline

---
## 📂 Step 1: Imports + Reproducibility

In [1]:
import os
import json
import math
import time
import random
import re
import unicodedata
import requests
import numpy as np
import torch
import ftfy
from tqdm import tqdm
from langdetect import detect, LangDetectException, DetectorFactory
from dotenv import load_dotenv

load_dotenv()

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
DetectorFactory.seed = 0

# NCBI API key
NCBI_API_KEY = '0c02b6e7f8e56927b51f9421816475b02e08'
RATE_LIMIT = 0.11 if NCBI_API_KEY else 0.34
BATCH_SIZE = 20

DATA_DIR = os.environ.get('DATA_DIR', '/workspace/data')

print(f'SEED: {SEED}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'PyTorch version: {torch.__version__}')
print(f'NCBI API key: {"✅ found" if NCBI_API_KEY else "❌ not set"}')
print(f'Data dir: {DATA_DIR}')


SEED: 42
CUDA available: True
PyTorch version: 2.10.0+cu128
NCBI API key: ✅ found
Data dir: /workspace/data


---
## 📂 Step 2: Load Data

In [2]:

os.environ['DATA_DIR'] = '/workspace/data'
data_dir = os.environ.get('DATA_DIR', './data')
DATASET_REGISTRY = {}

# Only load these datasets
ALLOWED_DATASETS = {'13B1_golden', '13B2_golden', '13B3_golden', '13B4_golden', 'training13b'}

print('📂 Loading standard JSON files...')
if not os.path.isdir(data_dir):
    print(f'❌ Directory not found: {data_dir}')
else:
    for filename in sorted(os.listdir(data_dir)):
        if filename.endswith('.json'):
            dataset_name = os.path.splitext(filename)[0]

            if dataset_name not in ALLOWED_DATASETS:  # ← filter here
                print(f'⏭️  Skipping: {filename}')
                continue

            file_path = os.path.join(data_dir, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    raw_data = json.load(f)
                    if isinstance(raw_data, dict) and 'questions' in raw_data:
                        data_list = raw_data['questions']
                    elif isinstance(raw_data, list):
                        data_list = raw_data
                    else:
                        data_list = [raw_data]
                    DATASET_REGISTRY[dataset_name] = {'data': data_list}
                except json.JSONDecodeError as e:
                    print(f'⚠️ Could not parse {filename}: {e}')

print('\n📋 Dataset Inventory:')
for name, meta in DATASET_REGISTRY.items():
    print(f'  [{name}] — {len(meta["data"])} documents')

📂 Loading standard JSON files...
⏭️  Skipping: pubmed_abstract_cache.json
⏭️  Skipping: training_final.json

📋 Dataset Inventory:
  [13B1_golden] — 85 documents
  [13B2_golden] — 85 documents
  [13B3_golden] — 85 documents
  [13B4_golden] — 85 documents
  [training13b] — 5389 documents


---
## 📂 Step 3: Cleaning Functions

In [3]:
import re
import ftfy
from langdetect import detect, LangDetectException
from tqdm import tqdm

def remove_html_tags(text: str) -> str:
    return re.sub(r'<[^>]+>', '', text)

def fix_encoding(text: str) -> str:
    return ftfy.fix_text(text)

def is_low_quality(text: str, min_words: int = 5, max_symbol_ratio: float = 0.3) -> bool:
    words = text.split()
    if len(words) < min_words:
        return True
    symbols = sum(1 for c in text if not c.isalnum() and not c.isspace())
    if symbols / max(len(text), 1) > max_symbol_ratio:
        return True
    return False

def detect_language(text: str, target_lang: str = 'en') -> bool:
    try:
        return detect(text) == target_lang
    except LangDetectException:
        return False

def remove_pii(text: str) -> str:
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.]+', '[EMAIL]', text)
    text = re.sub(r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b', '[PHONE]', text)
    text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[SSN]', text)
    return text

def strip_pubmed_metadata(text: str) -> str:
    # Extract abstract body — starts after section headers
    # e.g. INTRODUCTION:, BACKGROUND:, OBJECTIVE:, METHODS:, RESULTS:, ABSTRACT:
    abstract_match = re.search(
        r'(INTRODUCTION|BACKGROUND|OBJECTIVE|OBJECTIVES|SUMMARY|ABSTRACT|PURPOSE|AIM|AIMS)\s*:',
        text, flags=re.IGNORECASE
    )
    if abstract_match:
        text = text[abstract_match.start():]  # ← keep everything from here
    else:
        # Fallback: remove first 3 lines (citation + authors)
        lines = text.strip().splitlines()
        text = '\n'.join(lines[3:])

    # Remove remaining metadata
    text = re.sub(r'\b(doi|epub|pmid|pubmed)\b.*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.]+', '', text)
    text = re.sub(r'.*\b(university|hospital|department|institute|college|school|centre|center)\b.*',
                  '', text, flags=re.IGNORECASE | re.MULTILINE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_text(text: str) -> str | None:
    """Full cleaning pipeline for fetched abstract text."""
    text = strip_pubmed_metadata(text)  # ← add as first step
    text = remove_html_tags(text)
    text = fix_encoding(text)
    text = text.strip()
    if is_low_quality(text):
        return None
    if not detect_language(text):
        return None
    text = remove_pii(text)
    return text

def clean_cached_abstract(text: str) -> str | None:
    """Faster cleaning for PubMed abstracts — skips language detection."""
    text = strip_pubmed_metadata(text)
    text = remove_html_tags(text)
    text = fix_encoding(text)
    text = text.strip()
    if is_low_quality(text):
        return None
    text = remove_pii(text)
    return text

def clean_document(doc: dict) -> dict | None:
    text = doc.get('body', '')
    if not text and 'snippets' in doc:
        text = ' '.join([s.get('text', '') for s in doc['snippets']])
    text = remove_html_tags(text)
    text = fix_encoding(text)
    text = text.strip()
    if is_low_quality(text):
        return None
    if not detect_language(text):
        return None
    text = remove_pii(text)
    return {**doc, 'text': text}

# Apply cleaning
cleaned_datasets = {}
for name, meta in DATASET_REGISTRY.items():
    print(f'🧹 Cleaning dataset: {name} ({len(meta["data"])} documents)')
    cleaned_docs = []
    for doc in tqdm(meta['data'], desc=f'Cleaning {name}'):
        cleaned_doc = clean_document(doc)
        if cleaned_doc:
            cleaned_docs.append(cleaned_doc)
    cleaned_datasets[name] = cleaned_docs
    print(f'✅ Cleaned {len(cleaned_docs)} documents from {name}')

🧹 Cleaning dataset: 13B1_golden (85 documents)


Cleaning 13B1_golden: 100% 85/85 [00:00<00:00, 138.72it/s]


✅ Cleaned 73 documents from 13B1_golden
🧹 Cleaning dataset: 13B2_golden (85 documents)


Cleaning 13B2_golden: 100% 85/85 [00:00<00:00, 329.31it/s]


✅ Cleaned 75 documents from 13B2_golden
🧹 Cleaning dataset: 13B3_golden (85 documents)


...

✅ Cleaned 76 documents from 13B3_golden
🧹 Cleaning dataset: 13B4_golden (85 documents)


Cleaning 13B4_golden: 100% 85/85 [00:00<00:00, 305.67it/s]


✅ Cleaned 70 documents from 13B4_golden
🧹 Cleaning dataset: training13b (5389 documents)


Cleaning training13b: 100% 5389/5389 [00:11<00:00, 449.43it/s]

✅ Cleaned 4804 documents from training13b


In [4]:
def avg_word_length(text: str) -> float:
    words = text.split()
    return sum(len(w) for w in words) / max(len(words), 1)

def bullet_density(text: str) -> float:
    lines = text.splitlines()
    bullet_lines = sum(1 for l in lines if l.strip().startswith(('•', '-', '*', '·')))
    return bullet_lines / max(len(lines), 1)

def passes_quality_filter(doc: dict, verbose: bool = False) -> bool:
    text = doc.get('text', '')
    if not text or len(text) < 20:
        if verbose: print(f'FAIL length: {len(text)}')
        return False
    awl = avg_word_length(text)
    if not (2.5 <= awl <= 15):
        if verbose: print(f'FAIL awl: {awl:.2f}')
        return False
    if bullet_density(text) > 0.6:
        if verbose: print('FAIL bullets')
        return False
    return True

quality_filtered = {}
for name, docs in cleaned_datasets.items():
    before = len(docs)
    filtered = [doc for doc in docs if passes_quality_filter(doc)]
    quality_filtered[name] = filtered
    print(f'[{name}] quality filter: {before} → {len(filtered)} docs')

print('\n✅ Quality filter complete')


[13B1_golden] quality filter: 73 → 73 docs
[13B2_golden] quality filter: 75 → 75 docs
[13B3_golden] quality filter: 76 → 76 docs
[13B4_golden] quality filter: 70 → 70 docs
[training13b] quality filter: 4804 → 4803 docs

✅ Quality filter complete


In [5]:
import unicodedata

def normalize_text(text: str, form: str = 'NFC') -> str:
    text = unicodedata.normalize(form, text)
    text = re.sub(r'\r\n|\r', '\n', text)
    text = re.sub(r'\t', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()
    return text

normalized_datasets = {}
for name, docs in quality_filtered.items():
    normalized = [{**doc, 'text': normalize_text(doc['text'])} for doc in docs]
    normalized_datasets[name] = normalized

training_raw_data = normalized_datasets['training13b']
test_raw_data = (
    normalized_datasets['13B1_golden'] +
    normalized_datasets['13B2_golden'] +
    normalized_datasets['13B3_golden'] +
    normalized_datasets['13B4_golden']
)

print('✅ Normalization complete')
print(f'Training docs: {len(training_raw_data)}')
print(f'Test docs:     {len(test_raw_data)}')
for name, docs in normalized_datasets.items():
    if docs:
        sample = docs[0]['text']
        print(f'\n[{name}] sample: {sample[:120]}...' if len(sample) > 120 else f'\n[{name}] sample: {sample}')

✅ Normalization complete
Training docs: 4803
Test docs:     294

[13B1_golden] sample: What proportion of colorectal cancer cases are not assignable to any of the consensus molecular subtype (CMS) groups?

[13B2_golden] sample: Which ensemble machine-learning framework has been developed harnessing UK biobank data?

[13B3_golden] sample: How many primary genetic associations were identified through pQTL mapping within the Pharma Proteomics Project?

[13B4_golden] sample: Should Zotiraciclib be used for glioblastoma?

[training13b] sample: Is Hirschsprung disease a mendelian or a multifactorial disorder?


---
## 📂 Step 4: Fetching documents

In [6]:
BATCH_SIZE = 20  # fetch 20 abstracts per request

def extract_pmid(url: str) -> str:
    """Extract PMID from PubMed URL."""
    return url.rstrip('/').split('/')[-1]

def parse_batch_response(response_text: str, pmids: list) -> dict:
    """Parse multi-abstract response and map back to PMIDs."""
    import re
    results = {}
    sections = re.split(r'\n(?=\d+\.\s)', response_text)
    for section in sections:
        section = section.strip()
        if not section:
            continue
        match = re.match(r'^(\d+)\.', section)
        if match:
            pmid = match.group(1)
            if pmid in pmids:
                results[pmid] = section
    # Fallback: assign full text if parsing fails
    if not results and pmids:
        for pmid in pmids:
            results[pmid] = response_text
    return results

def fetch_pubmed_batch(urls: list, api_key: str = None) -> dict:
    """Fetch multiple abstracts in one API call. Returns {url: abstract_text}."""
    try:
        url_to_pmid = {}
        for url in urls:
            pmid = extract_pmid(url)
            if pmid.isdigit():
                url_to_pmid[url] = pmid
        if not url_to_pmid:
            return {}
        pmids = list(url_to_pmid.values())
        params = {
            'db': 'pubmed',
            'id': ','.join(pmids),
            'rettype': 'abstract',
            'retmode': 'text'
        }
        if api_key:
            params['api_key'] = api_key
        response = requests.get(
            'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi',
            params=params,
            timeout=30
        )
        if response.status_code != 200:
            return {}
        pmid_to_text = parse_batch_response(response.text, pmids)
        return {
            url: pmid_to_text.get(pmid, response.text)
            for url, pmid in url_to_pmid.items()
        }
    except Exception as e:
        print(f'Batch fetch error: {e}')
        return {}

# Test with first 3 URLs
test_urls = [s.get('document', '') for s in training_raw_data[0].get('snippets', [])[:3]]
print(f'Testing batch fetch with {len(test_urls)} URLs...')
results = fetch_pubmed_batch(test_urls, NCBI_API_KEY)
for url, text in results.items():
    print(f'URL: {url}')
    print(f'Abstract ({len(text)} chars): {text[:200]}...')


Testing batch fetch with 3 URLs...
URL: http://www.ncbi.nlm.nih.gov/pubmed/15829955
Abstract (4860 chars): 1. Nature. 2005 Apr 14;434(7035):857-63. doi: 10.1038/nature03467.

A common sex-dependent mutation in a RET enhancer underlies Hirschsprung disease 
risk.

Emison ES(1), McCallion AS, Kashuk CS, Bush...
URL: http://www.ncbi.nlm.nih.gov/pubmed/15617541
Abstract (4860 chars): 1. Nature. 2005 Apr 14;434(7035):857-63. doi: 10.1038/nature03467.

A common sex-dependent mutation in a RET enhancer underlies Hirschsprung disease 
risk.

Emison ES(1), McCallion AS, Kashuk CS, Bush...
URL: http://www.ncbi.nlm.nih.gov/pubmed/12239580
Abstract (4860 chars): 1. Nature. 2005 Apr 14;434(7035):857-63. doi: 10.1038/nature03467.

A common sex-dependent mutation in a RET enhancer underlies Hirschsprung disease 
risk.

Emison ES(1), McCallion AS, Kashuk CS, Bush...


In [7]:
# Deduplicate URLs (avoid redundant fetches)
# Collect all unique PubMed URLs across all docs
all_urls = set()
for doc in training_raw_data:
    for snippet in doc.get('snippets', []):
        url = snippet.get('document', '')
        if url:
            all_urls.add(url)

print(f'Total docs:        {len(training_raw_data)}')
print(f'Unique PubMed URLs: {len(all_urls)}')

# Estimate time
est_seconds = len(all_urls) * RATE_LIMIT
est_minutes = est_seconds / 60
print(f'\nEstimated fetch time: {est_minutes:.1f} minutes')
print(f'(Based on {RATE_LIMIT}s/request × {len(all_urls)} unique URLs)')

Total docs:        4803
Unique PubMed URLs: 37596

Estimated fetch time: 68.9 minutes
(Based on 0.11s/request × 37596 unique URLs)


In [8]:
# Load Cache (skip re-fetching on reruns) ──────────────────
cache_path = os.path.join(DATA_DIR, 'pubmed_abstract_cache.json')

if os.path.exists(cache_path):
    with open(cache_path, 'r', encoding='utf-8') as f:
        abstract_cache = json.load(f)
    print(f'✅ Loaded {len(abstract_cache)} cached abstracts')
else:
    print('❌ No cache found — run next cell first')

✅ Loaded 37556 cached abstracts


---
## 📂 Step 5: Build enriched training data

In [9]:
# Build Enriched Documents
enriched_docs = []
skipped = 0

for doc in tqdm(training_raw_data, desc='Building enriched docs'):
    question = doc.get('body', '')
    snippets = doc.get('snippets', [])

    # Collect full abstract texts for each snippet URL
    full_texts = []
    source_urls = []
    pmids = []

    for snippet in snippets:
        url = snippet.get('document', '')
        if url in abstract_cache:
            cleaned = clean_cached_abstract(abstract_cache[url])  # ← faster
            if cleaned:
                full_texts.append(cleaned)
                source_urls.append(url)
                pmids.append(extract_pmid(url))
        else:
            fallback = clean_text(snippet.get('text', ''))  # ← keep original for fallback
            if fallback:
                full_texts.append(fallback)

    if not full_texts:
        skipped += 1
        continue

    enriched_docs.append({
        **doc,
        "text": " ".join(full_texts),        # joined for RAG retrieval
        "abstracts": [                        # ← individual mapping
            {"pmid": pmid, "text": text}
            for pmid, text in zip(pmids, full_texts)
        ],
        "question": question,
        "source_urls": source_urls,
        "pmids": pmids,
        "n_sources": len(source_urls)
    })

print(f'\n✅ Enriched docs: {len(enriched_docs)}')
print(f'⚠️  Skipped:       {skipped} (no usable text found)')

# Preview
sample = enriched_docs[0]
print(f'\nSample question: {sample["question"][:100]}')
print(f'Sources: {sample["n_sources"]}')
print(f'Text ({len(sample["text"])} chars): {sample["text"][:300]}...')

Building enriched docs: 100% 4803/4803 [53:44<00:00,  1.49it/s]  


✅ Enriched docs: 4803
⚠️  Skipped:       0 (no usable text found)

Sample question: Is Hirschsprung disease a mendelian or a multifactorial disorder?
Sources: 16
Text (449182 chars): OBJECTIVE: To analyze the prognostic factors of colorectal cancer patients with synchronous liver metastasis treated by simultaneous colorectal and liver resection. METHODS: The clinical and follow-up data of 44 colorectal cancer patients with synchronous liver metastases who underwent simultaneous ...


---
## 📂 Step 6: Zero-shot Categorization on Full Abstracts

In [10]:
from transformers import pipeline
from tqdm import tqdm
from collections import Counter
import torch
import gc

CATEGORIES = [
    "Genetics & Mutations",
    "Cancer & Oncology",
    "Pharmacology & Drugs",
    "Neurology & Brain",
    "Cardiology & Heart",
    "Infectious Disease",
    "Immunology",
    "Metabolism & Diabetes",
    "Rare Diseases",
    "Surgery & Procedures",
    "Diagnostics & Imaging",
    "Mental Health",
    "Molecular Biology",
    "Clinical Guidelines",
    "Other"
]

device = 0 if torch.cuda.is_available() else -1
print(f'Using device: {"cuda" if device == 0 else "cpu"}')

print("Loading classifier...")
classifier = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=device
)
print("✅ Classifier ready!")

# Classify all docs
for doc in tqdm(enriched_docs, desc="Classifying"):
    result = classifier(doc['text'][:256], CATEGORIES)
    doc['category'] = result['labels'][0]
    doc['confidence'] = round(result['scores'][0], 4)
    doc['topic_id'] = CATEGORIES.index(result['labels'][0])

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

cats = Counter(doc['category'] for doc in enriched_docs)
print(f'\n✅ Done! {len(cats)} categories')
print(f'{"Category":<40} {"Count":>6} {"%":>6}')
print('-' * 55)
for cat, count in cats.most_common():
    pct = count / len(enriched_docs) * 100
    print(f'{cat:<40} {count:>6} ({pct:.1f}%)')

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Loading classifier...


Loading weights: 100% 106/106 [00:00<00:00, 543.11it/s, Materializing param=pooler.dense.weight]                                     
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Classifier ready!


Classifying:   0% 10/4803 [00:03<13:09,  6.07it/s]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Classifying: 100% 4803/4803 [07:50<00:00, 10.20it/s]



✅ Done! 15 categories
Category                                  Count      %
-------------------------------------------------------
Pharmacology & Drugs                        690 (14.4%)
Other                                       520 (10.8%)
Immunology                                  460 (9.6%)
Infectious Disease                          452 (9.4%)
Rare Diseases                               426 (8.9%)
Surgery & Procedures                        424 (8.8%)
Genetics & Mutations                        422 (8.8%)
Molecular Biology                           357 (7.4%)
Cancer & Oncology                           344 (7.2%)
Cardiology & Heart                          193 (4.0%)
Neurology & Brain                           191 (4.0%)
Diagnostics & Imaging                       120 (2.5%)
Mental Health                               106 (2.2%)
Metabolism & Diabetes                        65 (1.4%)
Clinical Guidelines                          33 (0.7%)


In [11]:
from collections import Counter

cats = Counter(doc['category'] for doc in enriched_docs)
total = len(enriched_docs)

print(f'{"Category":<40} {"Count":>6} {"%" :>6}')
print('-' * 55)
for cat, count in cats.most_common():
    pct = count / total * 100
    print(f'{cat:<40} {count:>6} ({pct:.1f}%)')

print(f'\nTotal docs: {total}')
print(f'Total categories: {len(cats)}')

Category                                  Count      %
-------------------------------------------------------
Pharmacology & Drugs                        690 (14.4%)
Other                                       520 (10.8%)
Immunology                                  460 (9.6%)
Infectious Disease                          452 (9.4%)
Rare Diseases                               426 (8.9%)
Surgery & Procedures                        424 (8.8%)
Genetics & Mutations                        422 (8.8%)
Molecular Biology                           357 (7.4%)
Cancer & Oncology                           344 (7.2%)
Cardiology & Heart                          193 (4.0%)
Neurology & Brain                           191 (4.0%)
Diagnostics & Imaging                       120 (2.5%)
Mental Health                               106 (2.2%)
Metabolism & Diabetes                        65 (1.4%)
Clinical Guidelines                          33 (0.7%)

Total docs: 4803
Total categories: 15


---
## 📂 Step 7: Save final data in CSV and Json format

In [ ]:
# Save Final Data 
import json
import pandas as pd
import os

# Create output directory
parsed_docs_dir = os.path.join(DATA_DIR, 'parsed_docs')
os.makedirs(parsed_docs_dir, exist_ok=True)

# Save as JSON
output_train = os.path.join(DATA_DIR, 'training_final.json')
with open(output_train, 'w', encoding='utf-8') as f:
    json.dump(enriched_docs, f, indent=2, ensure_ascii=False)
print(f'✅ Saved {len(enriched_docs)} training docs to {output_train}')

# Save as CSV
csv_path = os.path.join(parsed_docs_dir, 'training_final.csv')
df = pd.DataFrame([{
    'question':   doc.get('body', ''),
    'text':       doc.get('text', ''),
    'category':   doc.get('category', ''),
    'topic_id':   doc.get('topic_id', ''),
    'confidence': doc.get('confidence', ''),
    'n_sources':  doc.get('n_sources', 0),
    'source_urls': ', '.join(doc.get('source_urls', [])),
    'pmids':      ', '.join(doc.get('pmids', [])),
    'abstracts':  json.dumps(doc.get('abstracts', [])),
} for doc in enriched_docs])

df.to_csv(csv_path, index=False, encoding='utf-8')
print(f'✅ Saved CSV to {csv_path}')
print(f'   Rows:    {len(df)}')
print(f'   Columns: {list(df.columns)}')

# Verify
with open(output_train, 'r', encoding='utf-8') as f:
    verify = json.load(f)
print(f'\n✅ Verified JSON: {len(verify)} docs')
print(f'   Fields:   {list(verify[0].keys())}')
print(f'   Category: {verify[0].get("category")}')
print(f'   Sources:  {verify[0].get("n_sources")}')
print(f'   Text:     {verify[0]["text"][:150]}...')

✅ Saved 4803 training docs to /workspace/data/training_final.json
✅ Saved CSV to /workspace/data/parsed_docs/training_final.csv
   Rows:    4803
   Columns: ['question', 'text', 'category', 'topic_id', 'confidence', 'n_sources', 'source_urls', 'pmids', 'abstracts']


In [6]:
# Create output directory
import json
import pandas as pd
import os
parsed_docs_dir = os.path.join(DATA_DIR, 'parsed_docs')
os.makedirs(parsed_docs_dir, exist_ok=True)

# Save test data as JSON
output_test = os.path.join(DATA_DIR, 'test_final.json')
with open(output_test, 'w', encoding='utf-8') as f:
    json.dump(test_raw_data, f, indent=2, ensure_ascii=False)
print(f'✅ Saved {len(test_raw_data)} test docs to {output_test}')

# Save test as CSV
test_csv_path = os.path.join(parsed_docs_dir, 'test_final.csv')
df_test = pd.DataFrame([{
    'question':   doc.get('body', ''),
    'text':       doc.get('text', ''),
    'category':   doc.get('category', ''),
    'topic_id':   doc.get('topic_id', ''),
    'confidence': doc.get('confidence', ''),
    'n_sources':  doc.get('n_sources', 0),
    'source_urls': ', '.join(doc.get('source_urls', [])),
    'pmids':      ', '.join(doc.get('pmids', [])),
    'abstracts':  json.dumps(doc.get('abstracts', [])),
} for doc in test_raw_data])

df_test.to_csv(test_csv_path, index=False, encoding='utf-8')
print(f'✅ Saved test CSV to {test_csv_path}')
print(f'   Rows:    {len(df_test)}')

✅ Saved 294 test docs to /workspace/data/test_final.json
✅ Saved test CSV to /workspace/data/parsed_docs/test_final.csv
   Rows:    294


In [8]:
import pandas as pd

pd.set_option('display.max_colwidth', 100)

df = pd.read_csv('/workspace/data/parsed_docs/training_final.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (4803, 9)


,question,text,category,topic_id,confidence,n_sources,source_urls,pmids,abstracts
0,Is Hirschsprung disease a mendelian or a multifactorial disorder?,OBJECTIVE: To analyze the prognostic factors of colorectal cancer patients with synchronous live...,Surgery & Procedures,9,0.3225,16,"http://www.ncbi.nlm.nih.gov/pubmed/15829955, http://www.ncbi.nlm.nih.gov/pubmed/15617541, http:/...","15829955, 15617541, 12239580, 12239580, 15829955, 6650562, 15829955, 20598273, 21995290, 1585823...","[{""pmid"": ""15829955"", ""text"": ""OBJECTIVE: To analyze the prognostic factors of colorectal cancer..."
1,List signaling molecules (ligands) that interact with the receptor EGFR?,PURPOSE: Histone deacetylase (HDAC) inhibitors have shown promising clinical activity in the tre...,Pharmacology & Drugs,2,0.1407,15,"http://www.ncbi.nlm.nih.gov/pubmed/24323361, http://www.ncbi.nlm.nih.gov/pubmed/24124521, http:/...","24323361, 24124521, 23959273, 23888072, 23821377, 23787814, 23729230, 23399900, 23382875, 232129...","[{""pmid"": ""24323361"", ""text"": ""PURPOSE: Histone deacetylase (HDAC) inhibitors have shown promisi..."
2,Is the protein Papilin secreted?,"INTRODUCTION: Insertion of a ventriculoperitoneal (VP) shunt, the method of choice in the treatm...",Infectious Disease,5,0.2186,8,"http://www.ncbi.nlm.nih.gov/pubmed/21784067, http://www.ncbi.nlm.nih.gov/pubmed/19297413, http:/...","21784067, 19297413, 15094122, 15094110, 12666201, 11076767, 7515725, 3320045","[{""pmid"": ""21784067"", ""text"": ""INTRODUCTION: Insertion of a ventriculoperitoneal (VP) shunt, the..."
3,Are long non coding RNAs spliced?,"BACKGROUND: Imatinib mesylate, an orally administered kinase inhibitor that targets the Kit (CD1...",Molecular Biology,12,0.3204,6,"http://www.ncbi.nlm.nih.gov/pubmed/22955988, http://www.ncbi.nlm.nih.gov/pubmed/22955974, http:/...","22955988, 22955974, 22707570, 21622663, 24285305, 24106460","[{""pmid"": ""22955988"", ""text"": ""BACKGROUND: Imatinib mesylate, an orally administered kinase inhi..."
4,Is RANKL secreted from the cells?,BACKGROUND: Chronic cerebrospinal venous insufficiency (CCSVI) has been proposed to be associate...,Surgery & Procedures,9,0.2285,11,"http://www.ncbi.nlm.nih.gov/pubmed/24267510, http://www.ncbi.nlm.nih.gov/pubmed/24265865, http:/...","24267510, 24265865, 23835909, 23827649, 23698708, 23698708, 23632157, 22948539, 22901753, 228677...","[{""pmid"": ""24267510"", ""text"": ""BACKGROUND: Chronic cerebrospinal venous insufficiency (CCSVI) ha..."
